# 앨범 초동 판매량 예측 - CatBoost

- 실험 방법: **결측 대체 없는 원본 / 의미적(구조적) 결측 대체 / PCA** × **5-fold / 10-fold**
- 지표: RMSE, MAE, R² (n fold 평균 ± 표준편차)
- 이후 **최종 모델**(원본/CatBoost 네이티브 결측 처리 + 5-fold 기준 튜닝된 하이퍼파라미터)을 전체 데이터로 학습하고,
  반복 K-Fold로 표준오차/95% CI를 계산한 뒤 SHAP/PDP로 해석합니다.

> 원본 실험 노트북(`modeling.ipynb`)과 최종 모델 노트북(`catboost_final_model.ipynb`, Google Colab에서 작성)을
> 하나로 병합하면서, Colab 전용 설정(드라이브 마운트, 파일 업로드, 폰트 설치 등)과 두 노트북에서 중복되던
> 설정 코드는 제거하고 로컬/GitHub에서 바로 실행 가능한 형태로 정리했습니다.

In [ ]:
# 패키지 설치
!pip install -q catboost optuna shap scikit-learn openpyxl pandas

In [ ]:
# 데이터 경로 (이 저장소 기준 상대 경로)
DATA_PATH = 'data/fnc_final_target.xlsx' 

In [ ]:
# import 및 설정
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.inspection import PartialDependenceDisplay
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from catboost import CatBoostRegressor, Pool
import optuna
import shap

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 한글 폰트 설정 (OS별로 사용 가능한 폰트가 다르므로 순서대로 시도, 없으면 기본 폰트 사용)
import matplotlib.font_manager as fm

def set_korean_font():
    candidates = ['Malgun Gothic', 'AppleGothic', 'NanumGothic']
    available = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in available:
            plt.rcParams['font.family'] = name
            return name
    return None  # 한글 폰트가 없으면 라벨이 깨질 수 있음 (그래프 자체는 정상 출력)

set_korean_font()
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 공통 설정 (두 실험 단계에서 공유)
TARGET       = "실제 초동 판매량"
RANDOM_STATE = 42
N_TRIALS     = 20   # Optuna 시도 횟수

ID_COLS     = ["앨범명", "그룹명", "발매일"]
SNS_COLS    = ["Spotify 월간 청취자", "Spotify 팔로워", "TikTok 팔로워", "구독자", "Instagram 팔로워"]
TEASER_COLS = ["유튜브 티저 조회수", "유튜브 티저 댓글 수", "유튜브 티저 좋아요 수"]
CAT_COLS    = ["앨범 타입_국내외", "앨범 타입_음반 종류", "성별", "그룹 타입"]

# 원본/의미적 변형에서 제거할 SNS 컬럼 (Instagram 팔로워만 유지). PCA/최종 모델 변형은 제거하지 않음.
SNS_DROP_FOR_BASE = ["Spotify 팔로워", "Spotify 월간 청취자", "TikTok 팔로워", "구독자"]

def fixed_params():
    return {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": RANDOM_STATE,
        "thread_count": -1, # 모든 코어 사용
        "verbose": 0, # 콘솔 로그 출력 X
        "allow_writing_files": False,
    }

# 전처리 - 결측 대체 이전에 파생변수 생성
def add_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["티저_존재여부"]   = df["유튜브 티저 조회수"].notna().astype(int)
    df["데뷔앨범_여부"]   = df["컴백 주기"].isna().astype(int) # 컴백 주기 없으면 이전 활동 없음
    df["뮤비_존재여부"]   = df["직전 뮤비 최신 조회수"].notna().astype(int)
    df["빌보드_진입여부"] = df["직전 빌보드 200 순위"].notna().astype(int)
    return df

# 데이터는 이후 실험/최종 모델 단계에서 공통으로 사용
df = pd.read_excel(DATA_PATH)
print("shape:", df.shape)

## 1. 실험: 결측 대체 방식 & fold 수 비교

In [ ]:
def build_dataset(df: pd.DataFrame, variant: str):

    df = add_flags(df)
    y = df[TARGET].copy()
    X = df.drop(columns=ID_COLS + [TARGET])

    # 직전 초동 판매량 결측(직전 앨범 없는 데뷔앨범)일 경우 0 대체
    X["직전 초동 판매량"] = X["직전 초동 판매량"].fillna(0)

    pca_groups = None

    if variant in ("original", "semantic"):
        # 다중공선성 - sns 관련 칼럼 5가지 중 Instagram 팔로워만 유지
        X = X.drop(columns=SNS_DROP_FOR_BASE)

        if variant == "semantic":

            # 의미적 결측 처리 -> catboost 네이티브 결측 처리
            for c in TEASER_COLS:
                X[c] = X[c].fillna(0)
            X["이전 컴백기간 콘서트 횟수"] = X["이전 컴백기간 콘서트 횟수"].fillna(0)
            X["이전 컴백기간 최대규모 콘서트 관객수"] = X["이전 컴백기간 최대규모 콘서트 관객수"].fillna(0)
            X["직전 뮤비 최신 조회수"] = X["직전 뮤비 최신 조회수"].fillna(0)
            X["컴백 주기"] = X["컴백 주기"].fillna(0)
            X["직전 빌보드 200 순위"] = X["직전 빌보드 200 순위"].fillna(201)

    elif variant == "pca":
        # pca의 경우 sns 피처(5개), teaser 피처(3개) 유지해 fold 내부 PCA 차원 축소 (5->2, 3->2)
        pca_groups = {
            "SNS":    {"cols": SNS_COLS,    "n_components": 2},
            "TEASER": {"cols": TEASER_COLS, "n_components": 1},
        }
    else:
        raise ValueError(f"unknown variant: {variant}")

    cat_features = [c for c in CAT_COLS if c in X.columns]
    for c in cat_features:
        X[c] = X[c].astype(str) # str 변환

    return X, y, cat_features, pca_groups

In [ ]:
# fold별 PCA 진행
def apply_fold_pca(X_tr, X_va, pca_groups):

    # train fold로 fit하여 validation에 transform
    X_tr, X_va = X_tr.copy(), X_va.copy()
    for name, spec in pca_groups.items():

        cols, k = spec["cols"], spec["n_components"]
        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()
        pca = PCA(n_components=k, random_state=RANDOM_STATE)

        tr = pca.fit_transform(sca.fit_transform(imp.fit_transform(X_tr[cols])))
        va = pca.transform(sca.transform(imp.transform(X_va[cols])))

        pc_names = [f"{name}_PC{i+1}" for i in range(k)]
        X_tr = X_tr.drop(columns=cols)
        X_va = X_va.drop(columns=cols)
        X_tr[pc_names] = tr
        X_va[pc_names] = va

    return X_tr, X_va

In [ ]:
# 교차검증
def cross_validate(X, y, cat_features, params, n_folds, pca_groups=None, seed=RANDOM_STATE):

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    rmses, maes, r2s = [], [], []

    for tr_idx, va_idx in kf.split(X):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        if pca_groups is not None:
            X_tr, X_va = apply_fold_pca(X_tr, X_va, pca_groups)

        model = CatBoostRegressor(**params)
        model.fit(Pool(X_tr, y_tr, cat_features=cat_features), verbose=0)

        pred = model.predict(X_va)
        rmses.append(np.sqrt(mean_squared_error(y_va, pred)))
        maes.append(mean_absolute_error(y_va, pred))
        r2s.append(r2_score(y_va, pred))

    return {
        "rmse_mean": np.mean(rmses), "rmse_std": np.std(rmses),
        "mae_mean":  np.mean(maes),  "mae_std":  np.std(maes),
        "r2_mean":   np.mean(r2s),   "r2_std":   np.std(r2s),
    }

In [ ]:
# Optuna 튜닝
def tune(X, y, cat_features, n_folds, pca_groups, n_trials=N_TRIALS):
    def objective(trial):
        params = {
            **fixed_params(),
            "iterations":          trial.suggest_int("iterations", 200, 1500),
            "learning_rate":       trial.suggest_float("learning_rate", 1e-2, 3e-1, log=True),
            "depth":               trial.suggest_int("depth", 4, 10),
            "l2_leaf_reg":         trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
            "random_strength":     trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "border_count":        trial.suggest_int("border_count", 32, 255),
            "min_data_in_leaf":    trial.suggest_int("min_data_in_leaf", 1, 30),
        }
        return cross_validate(X, y, cat_features, params, n_folds, pca_groups)["rmse_mean"]

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return {**fixed_params(), **study.best_params}

In [ ]:
# 실험 실행
def run_experiment(df, variant, n_folds, label, n_trials=N_TRIALS):

    X, y, cat_features, pca_groups = build_dataset(df, variant)
    best_params = tune(X, y, cat_features, n_folds, pca_groups, n_trials=n_trials)
    res = cross_validate(X, y, cat_features, best_params, n_folds, pca_groups)
    res["label"] = label
    res["best_params"] = {k: v for k, v in best_params.items() if k not in fixed_params()}
    return res

# 평균 + 표준오차 형태
def fmt(mean, std):
    return f"{mean:,.3f} ± {std:,.3f}  ({mean-std:,.3f} ~ {mean+std:,.3f})"

In [ ]:
# 실험
experiments = [
    ("original", 5,  "원본 (대체 없음) / 5-fold"),
    ("semantic", 5,  "의미적 결측 대체 / 5-fold"),
    ("pca",      5,  "PCA (SNS5, 티저3) / 5-fold"),
    ("original", 10, "원본 (대체 없음) / 10-fold")
]

results = []
for variant, n_folds, label in experiments:
    print(f"[실행] {label}")
    results.append(run_experiment(df, variant, n_folds, label))

print("\n" + "=" * 70)
for r in results:
    print(f"\n■ {r['label']}")
    print(f"  RMSE : {fmt(r['rmse_mean'], r['rmse_std'])}")
    print(f"  MAE  : {fmt(r['mae_mean'],  r['mae_std'])}")
    print(f"  R^2  : {fmt(r['r2_mean'],   r['r2_std'])}")
    print(f"  best : {r['best_params']}")

In [ ]:
table = pd.DataFrame([{
    "실험": r["label"],
    "RMSE": f"{r['rmse_mean']:,.0f} ± {r['rmse_std']:,.0f}",
    "MAE":  f"{r['mae_mean']:,.0f} ± {r['mae_std']:,.0f}",
    "R^2":  f"{r['r2_mean']:.3f} ± {r['r2_std']:.3f}",
} for r in results])
table

## 2. 최종 모델

- **원본(결측 대체 없음) 변형 + CatBoost 네이티브 결측 처리**, 5-fold 기준으로 이전에 튜닝한 하이퍼파라미터를 사용해
  전체 데이터로 최종 모델을 학습합니다.
- 원본 작업(Google Colab)에서는 이 하이퍼파라미터를 Google Drive의 `best_numbers.json` 파일에서 불러왔지만,
  이 저장소에서는 재현성을 위해 그 값을 코드에 직접 명시했습니다.

In [ ]:
# 이전 Optuna 튜닝(원본 변형, 5-fold)에서 얻은 최적 하이퍼파라미터
BEST_NOPCA = {
    'iterations': 790,
    'learning_rate': 0.037823358901869525,
    'depth': 10,
    'l2_leaf_reg': 1.1220651489596645,
    'random_strength': 7.7858201199286805,
    'bagging_temperature': 0.39901381860667284,
    'border_count': 125,
    'min_data_in_leaf': 22,
}
print("best params:", BEST_NOPCA)

In [ ]:
# 최종 모델용 전처리 (SNS 5개 컬럼 모두 유지 - 원본 변형과 달리 다중공선성 컬럼을 드롭하지 않음)
def build_full_dataset(df):
    df = add_flags(df) # 더미 변수 추가
    y = df[TARGET].copy() # y
    X = df.drop(columns=ID_COLS + [TARGET])

    # 직전 초동 판매량 결측(직전 앨범 없는 데뷔앨범)일 경우 0 대체
    X["직전 초동 판매량"] = X["직전 초동 판매량"].fillna(0)

    cat_features = [c for c in CAT_COLS if c in X.columns]
    for c in cat_features:
        X[c] = X[c].astype(str)
    return X, y, cat_features

X, y, cat_features = build_full_dataset(df)
print("shape:", X.shape, "| 범주형:", cat_features)

In [ ]:
params = {**fixed_params(), **BEST_NOPCA}
model = CatBoostRegressor(**params)
model.fit(Pool(X, y, cat_features=cat_features), verbose=0)
print("학습 완료. 피처 수:", X.shape[1])

In [ ]:
# [반복 CV -> 표준오차/95% CI]
# Optuna 재튜닝 없음. 확정된 BEST_NOPCA 를 '평가'만 함.
# 프로토콜: K-fold 서로 다른 시드로 10회 반복 -> (K × 10)개 fold 값 수집.

def repeated_cv_collect(n_splits, n_repeats=10, seed=42, log=True):
    rkf = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed)
    rmses, maes, r2s = [], [], []
    t0 = time.time()
    for i, (tr, va) in enumerate(rkf.split(X)):
        m = CatBoostRegressor(**params)
        m.fit(Pool(X.iloc[tr], y.iloc[tr], cat_features=cat_features), verbose=0)
        pred = m.predict(X.iloc[va])
        rmses.append(np.sqrt(mean_squared_error(y.iloc[va], pred)))
        maes.append(mean_absolute_error(y.iloc[va], pred))
        r2s.append(r2_score(y.iloc[va], pred))
        if log and (i + 1) % n_splits == 0:
            print(f"  {n_splits}-fold 반복 {(i+1)//n_splits}/{n_repeats} 완료 "
                  f"| 누적 {i+1}개 | {time.time()-t0:.0f}s", flush=True)
    return np.array(rmses), np.array(maes), np.array(r2s)

def make_table(rmse_vals, mae_vals, r2_vals, label):
    def summarize(vals):
        m, sd = vals.mean(), vals.std(ddof=1)
        se = sd / np.sqrt(len(vals))
        return m, sd, se, m - 2*se, m + 2*se
    def fmt2(v, r2): return f"{v:.3f}" if r2 else f"{v:,.0f}"

    out = {}
    for name, vals in [("평균 RMSE", rmse_vals), ("평균 MAE", mae_vals), ("평균 R^2", r2_vals)]:
        r2 = "R^2" in name
        m, sd, se, lo, hi = summarize(vals)
        out[name] = {
            "평균±SE":      f"{fmt2(m, r2)} ± {fmt2(se, r2)}",
            "95% CI(±2SE)": f"({fmt2(lo, r2)} ~ {fmt2(hi, r2)})",
            "(참고)SD":     fmt2(sd, r2),
        }
    df_out = pd.DataFrame(out).T
    df_out.columns = pd.MultiIndex.from_product([[label], df_out.columns])
    return df_out

# 10-fold × 10회
print("[10-fold × 10회] 진행")
r10 = repeated_cv_collect(n_splits=10, n_repeats=10, seed=42)
tbl_10 = make_table(*r10, "CatBoost 10fold×10")
print(tbl_10.to_string())

In [ ]:
# 변수 중요도
imp = (pd.Series(model.get_feature_importance(), index=X.columns)
         .sort_values(ascending=False))
print("[변수 중요도]")
print(imp.to_string())

plt.figure(figsize=(8, 6))
imp.head(15).iloc[::-1].plot(kind="barh")
plt.title("CatBoost Feature Importance")
plt.xlabel("importance")
plt.tight_layout(); plt.show()

In [ ]:
# Partial Dependency Plot
#  세로축 = 그 변수만 바꿨을 때 예측 초동 판매량의 평균
#  가로축 = 실제 단위(팔로워 수/조회수/트랙 수 등)
#  우상향 = 변수 증가 -> 초동 예측 증가 / 역U자 = 특정 구간에서 최대, 하단 rug = 실제 데이터 분포

# 중요도 상위 수치형 12개
pdp_feats = [f for f in imp.index if f not in cat_features][:12]
print("[PDP 대상]", pdp_feats)

disp = PartialDependenceDisplay.from_estimator(model, X, pdp_feats, n_cols=3)
disp.figure_.set_size_inches(15, 12)
disp.figure_.suptitle("Partial Dependence", fontsize=14)
disp.figure_.tight_layout()
plt.show()

In [ ]:
#  summary(beeswarm)
pool = Pool(X, y, cat_features=cat_features)
shap_all = model.get_feature_importance(pool, type="ShapValues")
expected_value = shap_all[0, -1]     # 마지막 열 = base value(기대 예측값)
shap_matrix    = shap_all[:, :-1]    # 앞부분만 = 변수별 기여

# beeswarm summary
shap.summary_plot(shap_matrix, X, show=False)
plt.title("SHAP Summary (비-PCA, 전체데이터 단일모델)")
plt.tight_layout(); plt.show()

In [ ]:
# bar (mean |SHAP|) plot
shap.summary_plot(shap_matrix, X, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (mean |SHAP|)")
plt.tight_layout(); plt.show()

In [ ]:
# 특정 그룹(피원하모니) 예측 확인
mask = df["그룹명"].str.contains("P1Harmony", case=False, na=False)
print(f"피원하모니 앨범 (2020.12. ~ 2026.06.): {mask.sum()}")
if mask.sum() > 0:
    print(df.loc[mask, ["앨범명", "그룹명", "발매일", TARGET]].to_string())